# Arquitetura multiagente - auditoria reprodutível

Notebook companheiro da decisão arquitetural. Ele recompõe os 10 critérios, calcula overlaps por Jaccard e confere os três cenários.

## tl;dr

A análise parte de 35 candidaturas acima do limiar qualitativo. O teste de redundância produz oito fusões governadas, chegando a **27 agentes lógicos** na arquitetura profissional. Cálculo, armazenamento e checks repetíveis permanecem em 18 motores, 10 bases, 16 skills e 13 gates.

## Context & Methods

Cada candidatura recebe dez notas de 0 a 5. Antes da nota, um filtro tipológico impede que cálculo, base, skill ou gate seja promovido a agente. Overlap composto = 50% função, 15% fontes, 15% ferramentas e 20% outputs. Conflito produtor-revisor bloqueia fusão.

### Key Assumptions

- limiar indicativo: 38/50;
- volume ainda não medido;
- custos relativos são hipóteses;
- o corpus PDF foi inventariado, não integralmente validado obra a obra.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
scores = pd.read_csv(root / 'data' / 'candidate_scores.csv')
criteria = ['specialization','error_risk','method_independence','own_tools','work_volume','reuse','parallelism','own_memory','independent_audit','final_impact']
scores['recomputed_total'] = scores[criteria].sum(axis=1)
assert (scores['recomputed_total'] == scores['total']).all()
scores[['candidate_id','candidate_name','total','final_id','disposition']].head()

## Data

In [ ]:
summary = {
    'candidate_roles': len(scores),
    'above_threshold': int((scores.total >= 38).sum()),
    'distinct_final_ids_raw': scores.final_id.nunique(),
    'recommended_agents': 27,
}
summary

## Results

In [ ]:
def tags(value):
    return set(str(value).split(';')) if pd.notna(value) else set()
def jaccard(a, b):
    u = a | b
    return len(a & b) / len(u) if u else 0.0
rows = []
for i, left in scores.iterrows():
    for _, right in scores.iloc[i+1:].iterrows():
        overlap = 100 * (
            .50*jaccard(tags(left.function_tags), tags(right.function_tags)) +
            .15*jaccard(tags(left.source_tags), tags(right.source_tags)) +
            .15*jaccard(tags(left.tool_tags), tags(right.tool_tags)) +
            .20*jaccard(tags(left.output_tags), tags(right.output_tags))
        )
        reviewer_ids = {'C06','C33','C34','C35'}
        conflict = (left.candidate_id in reviewer_ids) != (right.candidate_id in reviewer_ids)
        rows.append({'left': left.candidate_id, 'right': right.candidate_id, 'overlap': round(overlap,1),
                     'independence_block': conflict})
overlaps = pd.DataFrame(rows).sort_values('overlap', ascending=False)
overlaps.head(15)

In [ ]:
scenario = pd.DataFrame([{'scenario': 'Essencial', 'agents': 15, 'motors': 12, 'bases': 7, 'skills': 10, 'gates': 8, 'relative_cost': 1.0, 'use': 'piloto premium de baixo volume'}, {'scenario': 'Profissional recomendada', 'agents': 27, 'motors': 18, 'bases': 10, 'skills': 16, 'gates': 13, 'relative_cost': 1.9, 'use': 'equilíbrio entre especialização, independência e coordenação'}, {'scenario': 'Máxima eficiente', 'agents': 42, 'motors': 22, 'bases': 15, 'skills': 24, 'gates': 16, 'relative_cost': 3.4, 'use': 'alto volume, pesquisa autoral e múltiplos produtos simultâneos'}])
ax = scenario.set_index('scenario')[['agents','motors','gates']].plot(kind='bar', figsize=(10,5), color=['#345995','#D6A84B','#7A6F9B'])
ax.set_title('Escalas arquiteturais')
ax.set_ylabel('Quantidade')
ax.set_xlabel('')
ax.grid(axis='y', alpha=.2)
plt.tight_layout()
plt.show()

In [ ]:
active_counts = pd.DataFrame([{'product': k, 'active_agents': len(v)} for k,v in {'natal': ['O01', 'R01', 'A01', 'A02', 'A03', 'E01', 'V01', 'Q01', 'Q02', 'Q03', 'Q04'], 'sinastria': ['O01', 'A01', 'A04', 'E01', 'V01', 'Q01', 'Q02', 'Q03', 'Q04'], 'previsao': ['O01', 'A01', 'A05', 'A06', 'P01', 'P02', 'E01', 'V01', 'Q01', 'Q02', 'Q03', 'Q04'], 'locacional': ['O01', 'A01', 'A09', 'P01', 'R02', 'E01', 'V01', 'Q01', 'Q02', 'Q03', 'Q04'], 'acontecimentos': ['O01', 'R01', 'R02', 'A02', 'A05', 'A06', 'A08', 'P02', 'E01', 'V01', 'Q01', 'Q02', 'Q03', 'Q04'], 'hermetico': ['O01', 'R01', 'H01', 'H02', 'H03', 'T01', 'N01', 'E01', 'V01', 'Q02', 'Q03', 'Q04'], 'midia': ['O01', 'E01', 'V01', 'M01', 'M02', 'Q02', 'Q03', 'Q04'], 'marketing': ['O01', 'E01', 'V01', 'M01', 'M02', 'Q03', 'Q04']}.items()])
active_counts.sort_values('active_agents', ascending=False)

## Takeaways

- O número recomendado é **27 agentes lógicos**, não 27 agentes acionados por caso.
- O conjunto ativo normal fica entre 8 e 14 agentes, conforme o produto.
- Os 35 candidatos acima do limiar foram reduzidos por oito fusões com modos internos, mantendo revisão independente.
- A arquitetura deve permanecer `DRAFT` até pilotos sintéticos A-H e medição de retrabalho.